In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
# Edits
import matplotlib as mpl
from mpl_toolkits.mplot3d import Axes3D
import pylcp
import pylcp.atom as atom
from pylcp.common import cart2spherical
from pylcp.fields import gaussianBeam
#plt.style.use('paper')

In [ ]:
gamma = 2*np.pi*30e6
t0 = 1e-5 # s
gammabar = gamma*t0
# wb = 4.7 # mm

class clippedGaussianBeam(pylcp.laserBeam):
    def __init__(self, kvec, pol, beta, delta, r0=np.array([0.,0.,0.]), wdiv=1, w0=1, wcut=1000, **kwargs):
        if callable(kvec):
            raise TypeError('kvec cannot be a function for a Gaussian beam.')

        if callable(pol):
            raise TypeError('Polarization cannot be a function for a Gaussian beam.')

        # Save the constant values (might be useful):
        self.con_kvec = kvec
        self.con_kmag = np.linalg.norm(kvec)
        self.con_khat = kvec/self.con_kmag
        if isinstance(pol, int) or isinstance(pol, float):
            if pol>0:
                pol = np.array([1., -1j, 0.])/np.sqrt(2)
            else:
                pol = np.array([1., +1j, 0.])/np.sqrt(2)
        self.con_pol = pol

        # Save the parameters specific to the Gaussian beam:
        self.beta_max = beta    # central saturation parameter
        self.wavelength = 2*np.pi/(np.linalg.norm(kvec))  
        self.r0 = r0    # Position of focus
        self.wdiv = wdiv
        self.w0 = w0
        self.wcut = wcut
        self.wcutfract = self.wcut/self.w0
        self.zr = np.pi*self.wdiv**2/(self.wavelength)  # Rayleigh length

        # Define the global rotation matrix
        self.global_rotation_matrix()
        
        # Use super class to define delta(t):
        super().__init__(delta=delta, **kwargs)
   

    def global_rotation_matrix(self):
        th = np.arccos(self.con_khat[2])
        ph = np.arctan2(self.con_khat[1], self.con_khat[0])
        
        rz = np.array([[np.cos(ph),-np.sin(ph),0.], [np.sin(ph),np.cos(ph),0.], [0.,0.,1.]])
        ry = np.array([[np.cos(th),0.,np.sin(th)], [0.,1.,0.], [-np.sin(th),0.,np.cos(th)]])
        
        self.rmat = rz@ry@np.linalg.inv(rz)
        self.rmat_inv = np.linalg.inv(self.rmat)
        return self.rmat
    
    
    def define_rotation_matrix(self):
        # Angles of rotation:|
        th = np.arccos(self.con_khat[2])
        phi = np.arctan2(self.con_khat[1], self.con_khat[0])
        
        # Use scipy to define the rotation matrix
        self.rmat = Rotation.from_euler('ZY', [phi, th]).inv().as_matrix() 
        self.rmat_inv = np.linalg.inv(self.rmat)

        
    def local_parameters(self, R=np.array([0., 0., 0.]), t=0.):
        # Adjust offset:
        Rp = R - self.r0.reshape((3,) + (1,)*(R.ndim-1))
    
        # Rotate up to the z-axis where we can apply formulas:
        Rp = np.einsum('ij,j...->i...', self.rmat_inv, Rp)
        rho_sq=np.sum(Rp[:2]**2, axis=0)
       
        # The waist at the position of interest:
        w = self.w0*np.sqrt(1+(Rp[2]**2/self.zr**2)) # w0*sqrt(1+(z/zr).^2);
        
        # Return the intensity:
        I = self.beta_max*(self.w0**2)/(w**2)*np.exp(-2*rho_sq/w**2)*(np.sqrt(rho_sq)<w*self.wcutfract) #Beta*(w0./w).^2.*exp(-2*r.^2./w.^2).*(r<w*wcutfract);
        
        # Now calculate the local k-vector in cylindrical coordinates:
        kr = (np.sqrt(rho_sq))*(Rp[2])/(self.zr**2+Rp[2]**2) # r.*z./(zr^2+z.^2);
        kt = np.zeros(Rp[0].shape)
        kz = np.ones(Rp[0].shape)
        
        # Convert to Cartesian: 
        kx = (kr*Rp[0]+kt*Rp[1])/(np.sqrt(Rp[0]**2+Rp[1]**2+1e-100))
        ky = (kr*Rp[1]+kt*Rp[0])/(np.sqrt(Rp[0]**2+Rp[1]**2+1e-100))
        
        # Normalize:
        kxn=kx/np.sqrt(kx**2+ky**2+kz**2) # normalized
        kyn=ky/np.sqrt(kx**2+ky**2+kz**2)
        kzn=kz/np.sqrt(kx**2+ky**2+kz**2)
        
        # Put into full array:
        kn=np.array([kxn,kyn,kzn])
        
        # Think about a way to do this without having to this without the FOR loop:
        it = np.nditer([kn[0], kn[1], kn[2], None, None, None], op_dtypes=['float64', 'float64', 'float64', 'complex128', 'complex128', 'complex128'])
        for (kxn, kyn, kzn, px, py, pz) in it:       
            thn = np.arccos(kzn)
            phn = np.arctan2(kyn, kxn)
            
            rzn = np.array([[np.cos(phn), -np.sin(phn), 0.],
                            [np.sin(phn),  np.cos(phn), 0.],
                            [         0.,           0., 1.]])
            ryn = np.array([[np.cos(thn),  0., np.sin(thn)],
                            [0.,           1.,          0.],
                            [-np.sin(thn), 0., np.cos(thn)]])
            
            rmatn = rzn@ryn@np.linalg.inv(rzn)

            (px[...], py[...], pz[...]) = self.rmat@rmatn@np.transpose(self.con_pol)
            
        # Rotate back:
        k = np.einsum('ij,j...->i...', self.rmat, kn)*self.con_kmag
        
        return k, cart2spherical(np.array(it.operands[3:])), I, self.delta(t)
    
    
    def beta(self, R=np.array([0., 0., 0.]), t=0.):
        k, P, I, D = self.local_parameters(R, t)
        return I
    
    def pol(self, R=np.array([0., 0., 0.]), t=0.):
        k, P, I, D = self.local_parameters(R, t)
        return P
    
    def kvec(self, R=np.array([0., 0., 0.]), t=0.):
        k, P, I, D = self.local_parameters(R, t)
        return k

# Set up the laser beams with their appropriate characteristics
delta = -1.5*gammabar
beta = 10
sigma = 1.0

In [ ]:
k = np.array([1., 1., 1.])
k = k/np.linalg.norm(k)
laserBeam = clippedGaussianBeam(kvec=np.array([0., 0., 1.]), pol=np.array([1., 1.j, 0])/np.sqrt(2), beta=1.0*beta, delta=delta, r0=np.array([0., 0., 0.]),\
                               wdiv=2, w0=200, wcut=2)

# wdiv speed at which diverges
# w0 shape
# wcut is where we cut off

laserBeam.local_parameters(np.array([100., 10., 1.]), 0.)

In [ ]:
x_beta = 5
X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
                   np.linspace(-x_beta, x_beta, 101))
z_tests = [-5, 0, +5] # position

plt.figure("Laser Beams", figsize=(4, 1.5*6)) # 6 beams
plt.clf()
# pr = cProfile.Profile()

for ii, z_test in enumerate(z_tests):
    Z = z_test*np.ones(X.shape)
    Rt=np.array([X, Y, Z])

    #pr.enable()
    """it = np.nditer([X, Y, Z, None])
    for (x, y, z, beta) in it:
        beta[...] = laserBeam.beta(np.array([x, y, z]), 0.)
    BETA = it.operands[3]"""
    #pr.disable()
    
    #pr.enable()
    BETA = laserBeam.beta(Rt)
    #pr.disable()

    plt.subplot(1., len(z_tests), ii+1)
    plt.imshow(BETA, origin='lower',
               extent=(-x_beta, x_beta,
                       -x_beta, x_beta))
    #plt.clim((0, 1))
    plt.set_cmap('jet')
    # Make a cross-hair:
    plt.plot([0, 0], [-x_beta, x_beta],
             'w-', linewidth=0.25)
    plt.plot([-x_beta, x_beta], [0, 0],
             'w-', linewidth=0.25)

In [ ]:
x_beta = 5
X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
                   np.linspace(-x_beta, x_beta, 101))
z_tests = [0] # position

plt.figure("Laser Beams", figsize=(4, 4)) # 6 beams
plt.clf()
# pr = cProfile.Profile()

for ii, z_test in enumerate(z_tests):
    Z = z_test*np.ones(X.shape)
    Rt=np.array([X, Y, Z])
    
    #pr.enable()
    BETA = laserBeam.beta(Rt)
    #pr.disable()

    plt.subplot(1., len(z_tests), ii+1)
    plt.plot(X[0,:], BETA[50,:])

In [ ]:
fig = plt.figure(figsize=(10,5)) # Change the size of the plot
ax = fig.gca(projection='3d')

# Make the grid
x, y, z = np.meshgrid(np.arange(-3000., 3001., 200.),
                      np.arange(-1000., 1001., 200.),
                      np.arange(-10000., 10001., 1000.)) # Change the axis

Rt=np.array([x, y, z])

arr = laserBeam.kvec(Rt)*20000
kx = arr[0]
ky = arr[1]
kz = arr[2]

i = laserBeam.beta(Rt)

ax.quiver(x,y,z,i*kx,i*ky,i*kz)
#ax.view_init(90, 90) # change this to see different viewing angles

plt.show()

In [ ]:
fig = plt.figure(figsize=(10,10)) # Change the size of the plot
ax = fig.gca(projection='3d')

# Make the grid
x, y, z = np.meshgrid(np.arange(-600., 601., 200.),
                      np.arange(-600., 601., 200.),
                      np.arange(-600., 601., 200.)) # Change the axis

Rt=np.array([x, y, z])

arrb, arr1b, ib, deltaLB = laserBeam.local_parameters(Rt)
arr = arrb*20000
arr1 = arr1b*20000
i = ib/50

#arr = laserBeam.kvec(Rt)*20000
kx = arr[0]
ky = arr[1]
kz = arr[2]

#i = laserBeam.beta(Rt)/50

#arr1 = laserBeam.cartesian_pol(Rt)*20000
p0 = arr1[0]
p1 = arr1[1]
p2 = arr1[2]

#print(np.amax(np.sqrt(kx**2+ky**2+kz**2)))
print(np.amax(kx*np.conj(p0)+ky*np.conj(p1)+kz*np.conj(p2))) # Need to divide out because of the 20000 multiplied
#print(np.amax(p1))


#quiver3(r(:,1),r(:,2),r(:,3),Ip.*k(:,1),Ip.*k(:,2),Ip.*k(:,3),'k')
ax.quiver(x,y,z,i*kx,i*ky,i*kz, color='k')

#quiver3(r(:,1),r(:,2),r(:,3),Ip.*real(P(:,1)),Ip.*real(P(:,2)),Ip.*real(P(:,3)),'r')
ax.quiver(x,y,z,i*np.real(p0),i*np.real(p1),i*np.real(p2), color='r')

#quiver3(r(:,1),r(:,2),r(:,3),Ip.*imag(P(:,1)),Ip.*imag(P(:,2)),Ip.*imag(P(:,3)),'b')
ax.quiver(x,y,z,i*np.imag(p0),i*np.imag(p1),i*np.imag(p2), color='b')

ax.set_xlabel('$x$')
ax.set_ylabel('$y$')  
ax.set_zlabel('$z$')

ax.view_init(30, -37) # change this to see different viewing angles

plt.show()